<a href="https://colab.research.google.com/github/vunhankhanhmcs3-cmd/master-thesis-mooc-recommendation/blob/main/01_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import pandas as pd
import os
import pickle

In [ ]:
# 1. Cấu hình
drive.mount('/content/drive')

# Đường dẫn (Cần đảm bảo bạn đã upload file raw vào đây)
BASE_PATH = '/content/drive/MyDrive/01. THESIS/My suggestion/'
PROCESSED_PATH = os.path.join(BASE_PATH, 'data 07.11.2026/processed/')
RELATIONS_PATH = os.path.join(BASE_PATH, 'data 07.11.2026/raw/relations/')
if not os.path.exists(PROCESSED_PATH): os.makedirs(PROCESSED_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 2. Đọc file User-Course (TSV)
file_path = os.path.join(RELATIONS_PATH, 'user-course.json')
print(f"Đang đọc file: {file_path}")
try:
    df_interact = pd.read_csv(file_path, sep='\t', header=None, names=['user_id', 'course_id'])
    df_interact['label'] = 1
except Exception as e:
    print(f"Lỗi đọc file: {e}")

Đang đọc file: /content/drive/MyDrive/01. THESIS/My suggestion/data 07.11.2026/raw/relations/user-course.json


In [ ]:
# 3. Lọc K-Core (Giữ lại User/Course có ít nhất 10 tương tác)
def filter_k_core(df, k=10):
    print(f"Đang lọc K-Core (k={k})...")
    df_temp = df.copy()
    while True:
        user_counts = df_temp['user_id'].value_counts()
        course_counts = df_temp['course_id'].value_counts()
        valid_users = user_counts[user_counts >= k].index
        valid_courses = course_counts[course_counts >= k].index
        new_df = df_temp[df_temp['user_id'].isin(valid_users) & df_temp['course_id'].isin(valid_courses)]
        if len(new_df) == len(df_temp): break
        df_temp = new_df
    return df_temp

df_clean = filter_k_core(df_interact, k=10) # Có thể giảm k=5 nếu muốn nhiều dữ liệu hơn

Đang lọc K-Core (k=10)...


In [ ]:
# 4. Tạo ID số (Remapping)
user_to_id = {u: i for i, u in enumerate(df_clean['user_id'].unique())}
course_to_id = {c: i for i, c in enumerate(df_clean['course_id'].unique())}

df_clean['uid'] = df_clean['user_id'].map(user_to_id)
df_clean['cid'] = df_clean['course_id'].map(course_to_id)

In [ ]:
# Lưu df_clean vào file CSV
output_df_path = os.path.join(PROCESSED_PATH, 'interactions_final.csv')
df_clean.to_csv(output_df_path, index=False)
print(f"Đã lưu df_clean vào: {output_df_path}")

# Lưu user_map vào file pickle
output_user_map_path = os.path.join(PROCESSED_PATH, 'user_map.pkl')
with open(output_user_map_path, 'wb') as f:
    pickle.dump(user_to_id, f)
print(f"Đã lưu user_to_id vào: {output_user_map_path}")

# Lưu course_map vào file pickle
output_course_map_path = os.path.join(PROCESSED_PATH, 'course_map.pkl')
with open(output_course_map_path, 'wb') as f:
    pickle.dump(course_to_id, f)
print(f"Đã lưu course_to_id vào: {output_course_map_path}")

Đã lưu df_clean vào: /content/drive/MyDrive/01. THESIS/My suggestion/data 07.11.2026/processed/interactions_final.csv
Đã lưu user_to_id vào: /content/drive/MyDrive/01. THESIS/My suggestion/data 07.11.2026/processed/user_map.pkl
Đã lưu course_to_id vào: /content/drive/MyDrive/01. THESIS/My suggestion/data 07.11.2026/processed/course_map.pkl


In [ ]:
# Đọc lại interactions_final.csv để kiểm tra
loaded_df_clean = pd.read_csv(output_df_path)
print("Dữ liệu đã làm sạch (df_clean) sau khi tải lại:")
display(loaded_df_clean.head())
print(f"Kích thước DataFrame đã làm sạch: {loaded_df_clean.shape}")

# Đọc lại user_map.pkl để kiểm tra
with open(output_user_map_path, 'rb') as f:
    loaded_user_to_id = pickle.load(f)
print(f"Số lượng user_to_id đã ánh xạ: {len(loaded_user_to_id)}")

# Đọc lại course_map.pkl để kiểm tra
with open(output_course_map_path, 'rb') as f:
    loaded_course_to_id = pickle.load(f)
print(f"Số lượng course_to_id đã ánh xạ: {len(loaded_course_to_id)}")

Dữ liệu đã làm sạch (df_clean) sau khi tải lại:


,user_id,course_id,label,uid,cid
0,U_545306,C_course-v1:TsinghuaX+20430064_2X+sp,1,0,0
1,U_545306,C_course-v1:TsinghuaX+02070251X+2019_T1,1,0,1
2,U_545306,C_course-v1:TsinghuaX+0350161X_2015_T2+sp,1,0,2
3,U_545306,C_course-v1:TsinghuaX+60240013X+sp,1,0,3
4,U_545306,C_course-v1:TsinghuaX+01030132X+sp,1,0,4


Kích thước DataFrame đã làm sạch: (96579, 5)
Số lượng user_to_id đã ánh xạ: 6486
Số lượng course_to_id đã ánh xạ: 549
